# Shape: profile a generated domain and check it against its contract (Python notebook)

Run after `shape_generate.ipynb`. Reads every Delta table named in the domain's contract,
profiles them together (so keys and foreign keys between tables are detected), checks the profile
against the contract, optionally diffs it against a baseline, writes the artifacts to
`Files/<outputDir>/<domain>/<timestamp>/`, shows the HTML report and returns a compact JSON
result to the calling pipeline.

The exit value is `{domain, tables, rows, passed, violations, drifted, changes, artifactPath,
truncated, kernel}`. `passed` is false when any table breaks the contract: a row count that is not
the generated one, a missing or extra column, a null in a column the schema says is never null, a
value outside an enumerated set, a duplicated primary key.


In [ ]:
%%configure
{"vCores": 8}


In [ ]:
# Upload the wheel(s) to this notebook's built-in resources folder (Resources > builtin):
# the platform wheel (Rust kernel), sqllocks_shape-0.9.0-py3-none-any.whl (pure Python) and, for a domain,
# sqllocks_shape_domains-0.9.0-py3-none-any.whl. pip takes the platform wheel when one
# fits and falls back to the pure wheel; the exit value reports the kernel.
# Once the packages are on PyPI the same line works without the upload.
%pip install --find-links builtin "sqllocks-shape==0.9.0" "sqllocks-shape-domains==0.9.0"

In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
# Paths are relative to the lakehouse Files/ folder, or absolute.
domain = "retail"  # labels the artifacts: Files/<outputDir>/<domain>/<timestamp>/
contractPath = "shape/retail/contract.json"  # the multi-table contract shape_generate wrote
tablePrefix = ""  # the Delta tables are <tablePrefix><table>, as shape_generate named them
baselinePath = ""  # optional earlier .shape artifact to diff against
outputDir = "shape"
failOnDrift = False  # True: drift against the baseline also fails the gate

In [ ]:
import json
import os
from pathlib import Path

import shape
from shape.integrations.fabric import generation
from shape.integrations.run_folder import claim_run_folder
from shape.kernel.dispatch import get_kernel

KERNEL = get_kernel().NAME
print(f"Shape {shape.__version__}, kernel: {KERNEL}")

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"
MAX_LISTED = 100


def _as_bool(value) -> bool:
    """Pipeline parameters may arrive as strings."""
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


def _resolve(path: str) -> str:
    return path if os.path.isabs(path) else f"{FILES}/{path}"


failOnDrift = _as_bool(failOnDrift)
generation.check_name(str(domain), "domain")
if tablePrefix:
    generation.check_name(str(tablePrefix), "tablePrefix")
safe_name = str(domain)
out_parent = f"{outputDir.strip('/')}/{safe_name}"  # one new folder per run is created below it

In [ ]:
from deltalake import DeltaTable

# The contract names the tables: every one of them is read and profiled.
with open(_resolve(contractPath), encoding="utf-8") as fh:
    contract_tables = list(json.load(fh)["tables"])

sources = {}
for name in contract_tables:
    generation.check_name(name, "table")
    delta_dir = f"{LAKEHOUSE}/Tables/{tablePrefix}{name}"
    sources[name] = DeltaTable(delta_dir).to_pyarrow_table()
total_rows = sum(t.num_rows for t in sources.values())
profile = shape.profile(sources, name=str(domain))
print(f"Profiled {total_rows:,} rows in {len(sources)} tables")

In [ ]:
violations, changes, drifted = [], [], False

if contractPath:
    check_result = shape.check(profile, _resolve(contractPath))
    violations = list(check_result.violations)

if baselinePath:
    diff_result = shape.diff(shape.load(_resolve(baselinePath)), profile)
    drifted = bool(diff_result.drifted)
    changes = list(diff_result.changes)
    if drifted and failOnDrift:
        violations.append(
            {
                "column": "*",
                "rule": "drift",
                "expected": "no drift against baseline",
                "observed": f"{len(changes)} change(s)",
            }
        )

passed = not violations

In [ ]:
out_dir = claim_run_folder(Path(FILES) / out_parent)  # a folder no other run owns
out_rel = f"{out_parent}/{out_dir.name}"
shape_file = out_dir / f"{safe_name}.shape"
shape.save(profile, str(shape_file))
(out_dir / f"{safe_name}.html").write_text(profile.to_html(), encoding="utf-8")
(out_dir / f"{safe_name}.summary.json").write_text(json.dumps(profile.summary()), encoding="utf-8")

artifact_path = f"{out_rel}/{safe_name}.shape"  # relative to Files/, usable as baselinePath
print("Artifacts written to", out_dir)

In [ ]:
from IPython.display import HTML, display

display(HTML(profile.to_html()))

In [ ]:
result = {
    "domain": domain,
    "tables": {name: t.num_rows for name, t in sources.items()},
    "rows": total_rows,
    "passed": passed,
    "violations": violations[:MAX_LISTED],
    "drifted": drifted,
    "changes": changes[:MAX_LISTED],
    "artifactPath": artifact_path,
    "truncated": len(violations) > MAX_LISTED or len(changes) > MAX_LISTED,
    "kernel": KERNEL,
}
print(json.dumps(result, indent=2, default=str)[:4000])

In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result, default=str))